In [1]:
# ============================================================
# Cell 1 - Setup
# ============================================================

import sys

sys.path.append("..")

In [2]:
# ============================================================
# Cell 2 - Load Model & Dataset
# ============================================================

import torch
import torch.nn as nn

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from torchvision.models import resnet18
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    10
)

checkpoint = torch.load(
    "../models/resnet18_cifar10.pth",
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✓ Model Loaded")

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )

])

test_dataset = datasets.CIFAR10(

    root="../data",

    train=False,

    download=True,

    transform=transform

)

test_loader = DataLoader(

    test_dataset,

    batch_size=1,

    shuffle=True

)

print("✓ Dataset Loaded")

Device: mps
✓ Model Loaded
✓ Dataset Loaded


In [3]:
# ============================================================
# Cell 3 - Neural State Collector
# ============================================================

from src.utils.neural_state_collector import NeuralStateCollector

target_layer = model.layer4[-1]

collector = NeuralStateCollector(target_layer)

print("✓ Neural State Collector Ready")

✓ Neural State Collector Ready


In [4]:
# ============================================================
# Cell 4 - Load UAIRE Pipeline
# ============================================================

from src.core.pipeline import UAIREPipeline

pipeline = UAIREPipeline()

print("✓ UAIRE Pipeline Ready")

✓ UAIRE Pipeline Ready


In [5]:
# ============================================================
# Cell 5 - Get One Sample
# ============================================================

images, labels = next(iter(test_loader))

image = images[0].permute(1, 2, 0).numpy()

images = images.to(device)

print("Image Shape:", images.shape)

Image Shape: torch.Size([1, 3, 32, 32])


In [6]:
# ============================================================
# Cell 6 - Collect Neural State
# ============================================================

collector.register_hooks()

states = collector.collect(

    model,

    images,

    compute_gradients=True

)

collector.remove_hooks()

print("✓ Neural State Collected")

print()

print(states.keys())

✓ Neural State Collected

dict_keys(['logits', 'prediction', 'activations', 'gradients'])


In [7]:
# ============================================================
# Cell 7 - Run UAIRE Pipeline
# ============================================================

features = pipeline.extract(

    model=model,

    input_tensor=images,

    output=states["logits"],

    activations=states["activations"],

    gradients=states["gradients"],

    image=image

)

print("Pipeline Executed Successfully")


========== INPUT QUALITY DEBUG ==========
Shape      : (32, 32)
Dtype      : uint8
Min Pixel  : 0
Max Pixel  : 139
Pipeline Executed Successfully


In [8]:
# ============================================================
# Cell 8 - Display Reliability Features
# ============================================================

print("="*70)
print("UAIRE RELIABILITY FEATURE VECTOR")
print("="*70)

print()

print("Total Features :", len(features))

print()

for key, value in features.items():

    print(f"{key:<35}: {value}")

UAIRE RELIABILITY FEATURE VECTOR

Total Features : 32

confidence                         : 0.6670068502426147
entropy                            : 1.1009633541107178
prediction_margin                  : 0.4990876317024231
activation_mean                    : 0.5459606051445007
activation_std                     : 0.7388502359390259
activation_max                     : 4.067745685577393
activation_min                     : 0.0
activation_energy                  : 0.8429064154624939
activation_sparsity                : 0.275390625
dead_neuron_ratio                  : 0.275390625
positive_activation_ratio          : 0.724609375
activation_entropy                 : 5.536521911621094
gradient_mean                      : -0.0026106741279363632
gradient_std                       : 0.02763970196247101
gradient_max                       : 0.05504905804991722
gradient_min                       : -0.10362022370100021
gradient_norm                      : 0.6275903582572937
gradient_energy        